In [28]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. DATA OVERVIEW & LOADING
# =====================================================================
print("--- Step 1: Loading Datasets ---")
df = pd.read_csv("train.csv")
df1 = pd.read_csv("test.csv")

print(f"Original Train Shape: {df.shape}")
print(f"Original Test Shape: {df1.shape}\n")

# =====================================================================
# 2. CORRECTED DATA CLEANING
# =====================================================================
print("--- Step 2: Cleaning Datasets ---")

# Fix: Do NOT use dropna(subset=['Cabin']) because it deletes 77%+ of your rows!
# Instead, we will drop the 'Cabin' column completely later in the reduction step.

# --- Clean Train Dataset ---
# Impute numerical values with the median (robust to outliers)
median_val = df["Age"].median()
df["Age"] = df["Age"].fillna(median_val)

# Fill categorical columns with the mode
mode_embarked = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(mode_embarked)

# --- Clean Test Dataset ---
# Impute numerical values with the median
median_valf = df1["Age"].median()
df1["Age"] = df1["Age"].fillna(median_valf)

# Fill 1 missing Fare in the test set using the median
median_val_fare = df1["Fare"].median()
df1["Fare"] = df1["Fare"].fillna(median_val_fare)

print(f"Train Duplicates Found: {df.duplicated().sum()}")
print(f"Test Duplicates Found: {df1.duplicated().sum()}\n")

# =====================================================================
# 3. DATA INTEGRATION
# =====================================================================
print("--- Step 3: Integrating Datasets ---")
# Stack train and test vertically
combined_df = pd.concat([df, df1], axis=0, ignore_index=True)
print(f"Combined Data Shape (No rows lost!): {combined_df.shape}")

# For testing purposes, impute missing 'Survived' values from the test stack
median_survived = combined_df["Survived"].median()
combined_df["Survived"] = combined_df["Survived"].fillna(median_survived)
print()

# =====================================================================
# 4. DATA TRANSFORMATION
# =====================================================================
print("--- Step 4: Transforming Data ---")

# A. Categorical Encoding (Convert Text to Numbers)
combined_df["Sex"] = combined_df["Sex"].map({"male": 0, "female": 1})

# One-Hot Encoding for 'Embarked' (Creates Embarked_Q and Embarked_S columns)
combined_df = pd.get_dummies(
    combined_df, columns=["Embarked"], drop_first=True, dtype=int
)

# B. Log Transformation (Smooth out heavily skewed Fare data)
combined_df["Fare_Log"] = np.log1p(combined_df["Fare"])

# C. Feature Scaling (Bring Age and Fare into the same numeric range)
scaler = StandardScaler()
combined_df[["Age_Scaled", "Fare_Scaled"]] = scaler.fit_transform(
    combined_df[["Age", "Fare"]]
)

print("Data transformation complete.\n")

# =====================================================================
# 5. DATA REDUCTION
# =====================================================================
print("--- Step 5: Reducing Data ---")

# Strategy A: Feature Selection (Drop uninformative identifiers & raw scaled duplicates)
# This safely discards the messy 'Cabin' column without losing your passenger rows.
columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin", "Age", "Fare"]
reduced_df = combined_df.drop(columns=columns_to_drop)
print(f"Shape after manual feature selection drop: {reduced_df.shape}")

# Strategy B: Dimensionality Reduction via PCA
# Compressing the correlated numeric columns ('Age_Scaled', 'Fare_Scaled') into 1 component
pca = PCA(n_components=1)
reduced_df["Age_Fare_PCA"] = pca.fit_transform(
    reduced_df[["Age_Scaled", "Fare_Scaled"]]
)

# Remove the raw scaled columns now that they are compressed into the PCA feature
reduced_df = reduced_df.drop(columns=["Age_Scaled", "Fare_Scaled"])
print(f"Final completely reduced shape: {reduced_df.shape}\n")

# =====================================================================
# FINAL CLEAN PREVIEW
# =====================================================================
print("--- Final Cleaned, Integrated, Transformed, and Reduced Dataset ---")
print(reduced_df.head())

--- Step 1: Loading Datasets ---
Original Train Shape: (891, 12)
Original Test Shape: (418, 11)

--- Step 2: Cleaning Datasets ---
Train Duplicates Found: 0
Test Duplicates Found: 0

--- Step 3: Integrating Datasets ---
Combined Data Shape (No rows lost!): (1309, 12)

--- Step 4: Transforming Data ---
Data transformation complete.

--- Step 5: Reducing Data ---
Shape after manual feature selection drop: (1309, 10)
Final completely reduced shape: (1309, 9)

--- Final Cleaned, Integrated, Transformed, and Reduced Dataset ---
   Survived  Pclass  Sex  SibSp  Parch  Embarked_Q  Embarked_S  Fare_Log  \
0       0.0       3    0      1      0           0           1  2.110213   
1       1.0       1    1      1      0           0           0  4.280593   
2       1.0       3    1      0      0           0           1  2.188856   
3       1.0       1    1      1      0           0           1  3.990834   
4       0.0       3    0      0      0           0           1  2.202765   

   Age_Fare_PC